# Geometric Deep Learning: Symmetry, Groups, and Graph Neural Networks
## Equivariance as an architectural principle

**MAT 4953/6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/geometric_deep_learning.ipynb)

---

> **Prerequisites:** Linear algebra (matrices, linear maps), basic abstract algebra (groups are introduced from scratch), familiarity with feedforward networks and CNNs ([`dnn_architectures_overview.ipynb`](dnn_architectures_overview.ipynb)).

Why do convolutional neural networks work for images? The standard answer — *parameter sharing reduces model size* — is correct but misses the deeper reason. The true answer is that **CNNs encode a symmetry of the problem**: natural images are translation-equivariant — a cat in the top-left corner is the same cat in the bottom-right corner. The architecture *bakes in* this knowledge.

**Geometric deep learning** (Bronstein, Bruna, Cohen, Veličković, 2021) generalises this principle: *build symmetry into the architecture, not into the data*. The same framework unifies CNNs, graph neural networks, recurrent networks, and transformers as special cases of a single blueprint. The mathematical language is **group theory**.

**Outline**
1. [Symmetry and inductive bias](#part1)
2. [Groups, actions, equivariance](#part2)
3. [Graphs and permutation symmetry](#part3)
4. [Graph neural networks and GCNs](#part4)
5. [Expressivity: the Weisfeiler–Lehman test](#part5)
6. [Beyond 2D: 3D equivariance in science](#part6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from ipywidgets import interact, IntSlider, Dropdown
from scipy.signal import convolve2d

# networkx for graph drawing
try:
    import networkx as nx
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'networkx', '-q'])
    import networkx as nx

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 100})
rng = np.random.default_rng(0)
print(f"NetworkX {nx.__version__} loaded.")

<a id="part1"></a>
# Part 1: Symmetry as an Inductive Bias

## 1.1 Why symmetry matters

Consider training an image classifier. If the network has no built-in knowledge of translation symmetry, it must *learn* from data that a cat at position $(10,10)$ and a cat at position $(50,80)$ are the same object. That requires vastly more labelled examples. **Encoding the symmetry directly reduces the effective size of the hypothesis class**, improving sample efficiency.

The general principle:
- **Invariance:** $f(g \cdot \mathbf{x}) = f(\mathbf{x})$ — the output does not change under symmetry $g$ (e.g., a classifier should give the same label regardless of position).
- **Equivariance:** $f(g \cdot \mathbf{x}) = g \cdot f(\mathbf{x})$ — the output *transforms consistently* with the input (e.g., a feature map should shift when the image shifts).

Invariance and equivariance are related: invariance = equivariance followed by a pooling operation that discards the group action.

## 1.2 Convolution is equivariant to translation

The 2-D convolution $(f * h)[m,n] = \sum_{k,l} f[k,l]\,h[m-k,n-l]$ satisfies:
$$(\mathcal{T}_{(a,b)} f) * h = \mathcal{T}_{(a,b)}(f * h),$$
where $\mathcal{T}_{(a,b)}$ shifts the image by $(a,b)$. This is the **translation-equivariance** of convolution — the defining property of CNNs.

In [ ]:
# ── Demonstrate translation equivariance of convolution ─────────────────────
rng2 = np.random.default_rng(1)
img = np.zeros((20, 20))
img[6:10, 6:10] = 1.0           # a small square "object"

kernel = np.array([[1, 0, -1],
                   [2, 0, -2],
                   [1, 0, -1]], dtype=float)   # Sobel edge detector

def translate(arr, dr, dc):
    result = np.zeros_like(arr)
    r0, r1 = max(0, dr), min(arr.shape[0], arr.shape[0]+dr)
    c0, c1 = max(0, dc), min(arr.shape[1], arr.shape[1]+dc)
    rs0, rs1 = max(0, -dr), min(arr.shape[0], arr.shape[0]-dr)
    cs0, cs1 = max(0, -dc), min(arr.shape[1], arr.shape[1]-dc)
    result[r0:r1, c0:c1] = arr[rs0:rs1, cs0:cs1]
    return result

dr, dc = 4, 3
img_shifted = translate(img, dr, dc)

feat          = convolve2d(img,         kernel, mode='same', boundary='fill')
feat_of_shift = convolve2d(img_shifted, kernel, mode='same', boundary='fill')
shift_of_feat = translate(feat, dr, dc)

fig, axes = plt.subplots(2, 3, figsize=(11, 6))
data = [(img,          'Original $f$'),
        (img_shifted,  r'Shifted $\mathcal{T}f$'),
        (feat,         r'$f * h$  (conv of original)'),
        (feat_of_shift,r'$(\mathcal{T}f) * h$  (conv of shifted)'),
        (shift_of_feat,r'$\mathcal{T}(f * h)$  (shift of conv)'),
        (np.abs(feat_of_shift - shift_of_feat), r'$|(\mathcal{T}f)*h - \mathcal{T}(f*h)|$')]

for ax, (img_, title) in zip(axes.ravel(), data):
    ax.imshow(img_, cmap='RdBu_r', origin='upper',
              vmin=-1.5 if 'conv' in title.lower() else 0,
              vmax=1.5 if 'conv' in title.lower() else 1)
    ax.set_title(title, fontsize=10); ax.axis('off')

fig.suptitle('Convolution is equivariant to translation  '
             r'(bottom-right $\approx 0$ confirms equivariance)', fontsize=12)
plt.tight_layout(); plt.show()
print(f"Max deviation from perfect equivariance: {np.abs(feat_of_shift - shift_of_feat).max():.2e}")

<a id="part2"></a>
# Part 2: Groups, Actions, and Equivariance

## 2.1 Groups

A **group** is a set $G$ with a binary operation $\cdot$ satisfying:
1. *Closure:* $g \cdot h \in G$ for all $g, h \in G$.
2. *Associativity:* $(g \cdot h) \cdot k = g \cdot (h \cdot k)$.
3. *Identity:* $\exists\, e \in G$ such that $e \cdot g = g \cdot e = g$.
4. *Inverses:* $\forall\, g \in G$, $\exists\, g^{-1}$ such that $g \cdot g^{-1} = e$.

**Examples relevant to deep learning:**

| Group | Elements | Operation | Application |
|-------|---------|-----------|-------------|
| $(\mathbb{Z}^2, +)$ | 2-D integer vectors | Vector addition | Translation of image pixels |
| $SO(2)$ | $2\times2$ rotation matrices | Matrix multiplication | Rotation of 2-D images |
| $SO(3)$ | $3\times3$ rotation matrices | Matrix multiplication | Rotation of 3-D molecules |
| $E(3)$ | Rotations + translations in $\mathbb{R}^3$ | Composition | 3-D point clouds, proteins |
| $S_n$ | Permutations of $\{1,\ldots,n\}$ | Function composition | Sets, graphs (relabelling nodes) |

## 2.2 Group actions and equivariance

A **group action** of $G$ on a space $\mathcal{X}$ is a map $\rho: G \times \mathcal{X} \to \mathcal{X}$ (written $g \cdot \mathbf{x}$) satisfying $e \cdot \mathbf{x} = \mathbf{x}$ and $(g h) \cdot \mathbf{x} = g \cdot (h \cdot \mathbf{x})$.

A function $f: \mathcal{X} \to \mathcal{Y}$ is **$G$-equivariant** if:
$$\boxed{f(g \cdot \mathbf{x}) = g \cdot f(\mathbf{x}) \qquad \forall\, g \in G, \; \mathbf{x} \in \mathcal{X}.}$$

**Key theorem:** The space of $G$-equivariant linear maps $f: \mathbb{R}^n \to \mathbb{R}^n$ (where $G$ acts by permutation matrices) is exactly the space of matrices of the form $\alpha I + \beta \mathbf{1}\mathbf{1}^\top$ — *i.e.*, linear combinations of the identity and the all-ones matrix. Any layer of a permutation-equivariant network on sets must have this form!

In [ ]:
# ── Visualise: rotation is NOT a symmetry of a standard MLP ─────────────────
# A standard learned filter is not equivariant to rotation.
# We show this by applying a 45° rotation before vs. after a non-symmetric filter.

theta_rot = np.pi / 4    # 45 degrees

def rotate_img(arr, angle):
    from scipy.ndimage import rotate as sci_rotate
    return sci_rotate(arr, np.degrees(angle), reshape=False, order=1)

def apply_filter(arr, filt):
    return convolve2d(arr, filt, mode='same', boundary='fill')

# Asymmetric filter (not rotation-equivariant)
asym_filt = np.array([[2, 1, 0],
                       [1, 0,-1],
                       [0,-1,-2]], dtype=float) / 4.0

# Symmetric filter (rotation-equivariant for 90° rotations)
sym_filt = np.array([[1, 1, 1],
                      [1, 8, 1],
                      [1, 1, 1]], dtype=float) / 16.0

img_blob = np.zeros((24, 24))
img_blob[8:16, 4:10] = 1.0      # asymmetric object

img_rot  = rotate_img(img_blob, theta_rot)

fig, axes = plt.subplots(2, 4, figsize=(13, 6))
row_labels = ['Asymmetric filter\n(NOT equivariant)', 'Isotropic filter\n(equivariant to rotations)']
for row, filt in enumerate([asym_filt, sym_filt]):
    f_then_rot = rotate_img(apply_filter(img_blob, filt), theta_rot)
    rot_then_f = apply_filter(img_rot, filt)
    diff        = np.abs(f_then_rot - rot_then_f)

    for col, (im, ttl) in enumerate([
            (img_blob,    'Input $f$'),
            (img_rot,     r'Rotated input $\rho(g)f$'),
            (f_then_rot,  r'Filter $\to$ rotate: $\rho(g)(Wf)$'),
            (rot_then_f,  r'Rotate $\to$ filter: $W(\rho(g)f)$'),
    ]):
        axes[row][col].imshow(im, cmap='RdBu_r', origin='upper')
        axes[row][col].set_title(ttl, fontsize=9)
        axes[row][col].axis('off')

for row, lbl in enumerate(row_labels):
    axes[row][0].set_ylabel(lbl, fontsize=9, rotation=90, labelpad=4)

fig.suptitle('Equivariance to rotation depends on filter symmetry', fontsize=12)
plt.tight_layout(); plt.show()

<a id="part3"></a>
# Part 3: Graphs and Permutation Symmetry

## 3.1 Why graphs?

Many real-world objects are naturally represented as graphs:
- **Molecules:** atoms = nodes (with element type), chemical bonds = edges.
- **Social networks:** users = nodes, friendships = edges.
- **Particle physics:** particles = nodes, interactions = edges.
- **Knowledge graphs:** entities = nodes, relations = edges.

A graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ has $n = |\mathcal{V}|$ nodes and $m = |\mathcal{E}|$ edges. Node features are stored as a matrix $\mathbf{H} \in \mathbb{R}^{n \times d}$, and the graph structure is captured by the **adjacency matrix** $A \in \{0,1\}^{n \times n}$.

**The key symmetry:** The labels we assign to nodes are arbitrary. Permuting the nodes — applying any $\sigma \in S_n$ — should not change the prediction (for a graph-level task) or should permute the predictions consistently (for a node-level task). The architecture must be **permutation equivariant** (for node-level tasks) or **permutation invariant** (for graph-level tasks).

In [ ]:
# ── Draw some example graphs ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
graph_defs = [
    ('Benzene (C₆H₆)',
     list(range(6)),
     [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)],
     ['C']*6, '#3498db'),
    ('Karate club\n(social network)',
     list(range(8)),
     [(0,1),(0,2),(0,3),(1,4),(2,5),(3,6),(4,7),(5,7),(6,7),(1,2)],
     [str(i) for i in range(8)], '#e74c3c'),
    ('Knowledge graph\n(entity–relation)',
     ['Paris','France','EU','Macron','Person'],
     [('Paris','France'),('France','EU'),('Macron','France'),('Macron','Person'),('Paris','Person')],
     None, '#2ecc71'),
]

for ax, (title, nodes, edges, labels, color) in zip(axes, graph_defs):
    G_nx = nx.Graph()
    G_nx.add_nodes_from(nodes)
    G_nx.add_edges_from(edges)
    pos = nx.spring_layout(G_nx, seed=3)
    nx.draw_networkx_nodes(G_nx, pos, ax=ax, node_color=color,
                           node_size=400, alpha=0.85)
    nx.draw_networkx_edges(G_nx, pos, ax=ax, edge_color='#555555',
                           width=1.5, alpha=0.7)
    if labels:
        nx.draw_networkx_labels(G_nx, pos, ax=ax,
                                labels={n: l for n, l in zip(nodes, labels)},
                                font_size=9, font_color='white', font_weight='bold')
    else:
        nx.draw_networkx_labels(G_nx, pos, ax=ax, font_size=8)
    ax.set_title(title, fontsize=11); ax.axis('off')

fig.suptitle('Graphs arise naturally in chemistry, social networks, and knowledge representation',
             fontsize=12)
plt.tight_layout(); plt.show()

## 3.2 Message passing

The dominant framework for GNNs is **message passing** (Gilmer et al., 2017). Each node iteratively aggregates information from its neighbours:

$$\boxed{\mathbf{h}_v^{(k+1)} = \phi\!\left(\mathbf{h}_v^{(k)},\; \bigoplus_{u \in \mathcal{N}(v)} \psi\!\left(\mathbf{h}_v^{(k)},\, \mathbf{h}_u^{(k)}\right)\right),}$$

where:
- $\mathbf{h}_v^{(k)} \in \mathbb{R}^d$ is the feature vector of node $v$ at layer $k$,
- $\mathcal{N}(v)$ is the set of neighbours of $v$,
- $\psi$ is the **message function** (what each neighbour sends),
- $\bigoplus$ is a **permutation-invariant aggregation** (e.g., sum, mean, max),
- $\phi$ is the **update function** (combines own features with aggregated messages).

After $K$ layers, $\mathbf{h}_v^{(K)}$ captures information from the $K$-hop neighbourhood of $v$. A graph-level prediction uses a final **readout** $\mathbf{h}_{\mathcal{G}} = \mathrm{READOUT}(\{\mathbf{h}_v^{(K)}\}_{v \in \mathcal{V}})$, which must also be permutation-invariant (e.g., sum over all nodes).

In [ ]:
# ── Animate message passing on a small molecule graph ───────────────────────
# Molecule: ethanol (CH3-CH2-OH), 3 heavy atoms simplified to a chain
# Nodes: 0=C, 1=C, 2=O   Features: [atomic_num, H_count]
node_feats_init = np.array([[6, 3],   # C, 3 H
                              [6, 2],   # C, 2 H
                              [8, 1]], dtype=float)  # O, 1 H
adj = np.array([[0,1,0],
                [1,0,1],
                [0,1,0]], dtype=float)   # chain C-C-O

def message_pass_step(H, A):
    # GCN-style: H_new = D^{-1/2} A~ D^{-1/2} H  (with self-loops)
    A_tilde = A + np.eye(len(A))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(A_tilde.sum(axis=1)))
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
    return A_hat @ H

node_names   = ['C (CH₃)', 'C (CH₂)', 'O (OH)']
feat_names   = ['atomic #', '# H']
colors_atoms = ['#3498db', '#3498db', '#e74c3c']

G_mol = nx.Graph()
G_mol.add_edges_from([(0,1),(1,2)])
pos_mol = {0: (0,0), 1: (1.5,0), 2: (3,0)}

H_layers = [node_feats_init.copy()]
H = node_feats_init.copy()
for _ in range(3):
    H = message_pass_step(H, adj)
    H_layers.append(H.copy())

def _plot_mp(layer=0):
    H_show = H_layers[layer]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.0),
                              gridspec_kw={'width_ratios': [1, 1.5]})
    ax = axes[0]
    nx.draw_networkx_nodes(G_mol, pos_mol, ax=ax,
                           node_color=colors_atoms, node_size=900, alpha=0.9)
    nx.draw_networkx_edges(G_mol, pos_mol, ax=ax, edge_color='#555555', width=2.5)
    nx.draw_networkx_labels(G_mol, pos_mol, ax=ax,
                            labels={i: node_names[i] for i in range(3)},
                            font_size=8, font_color='white', font_weight='bold')
    # Show feature values above each node
    for i, (x, y) in pos_mol.items():
        ax.text(x, y + 0.35,
                f'[{H_show[i,0]:.2f}, {H_show[i,1]:.2f}]',
                ha='center', fontsize=9, color='#333333')
    ax.set_title(f'Layer {layer}: node features', fontsize=11)
    ax.set_xlim(-0.8, 3.8); ax.set_ylim(-0.8, 0.9); ax.axis('off')

    # Bar chart of features
    x_pos = np.arange(3)
    width = 0.35
    axes[1].bar(x_pos - width/2, H_show[:, 0], width, label='atomic #',
                color='#3498db', edgecolor='k', lw=0.5)
    axes[1].bar(x_pos + width/2, H_show[:, 1], width, label='# H',
                color='#e74c3c', edgecolor='k', lw=0.5)
    axes[1].set_xticks(x_pos); axes[1].set_xticklabels(node_names, fontsize=10)
    axes[1].set_title(f'Node features after {layer} message-passing steps', fontsize=11)
    axes[1].legend(fontsize=10)
    axes[1].set_ylabel('Feature value')
    plt.tight_layout(); plt.show()

interact(_plot_mp,
         layer=IntSlider(min=0, max=3, step=1, value=0, description='Layer:'));

<a id="part4"></a>
# Part 4: Graph Convolutional Networks

## 4.1 GCN — spectral motivation

The **Graph Convolutional Network** (Kipf & Welling, 2017) derives its update rule from spectral graph theory. Define the normalised adjacency with self-loops:
$$\hat{A} = \tilde{D}^{-1/2}\tilde{A}\,\tilde{D}^{-1/2}, \qquad \tilde{A} = A + I, \quad \tilde{D}_{ii} = \sum_j \tilde{A}_{ij}.$$

A GCN layer is:
$$\boxed{H^{(k+1)} = \sigma\!\left(\hat{A}\, H^{(k)}\, W^{(k)}\right),}$$
where $W^{(k)} \in \mathbb{R}^{d_k \times d_{k+1}}$ is a learnable weight matrix shared across all nodes.

**Permutation equivariance:** $\hat{A}$ and the multiplication $\hat{A} H W$ commute with any permutation $P$:
$$\hat{A}_{P} (P H) W = P (\hat{A} H W),$$
since permuting nodes permutes both $\hat{A}$ (via $P\hat{A}P^\top$) and $H$ (via $PH$) consistently. The layer is manifestly permutation-equivariant.

## 4.2 Spectral interpretation

The normalised Laplacian $\mathbf{L} = I - \hat{A}$ has eigenvectors $U$ (the graph Fourier basis). A spectral convolution with filter $\hat{g}$ is $U\,\mathrm{diag}(\hat{g})\,U^\top \mathbf{h}$. The GCN layer with $\hat{A} = I - \mathbf{L}$ corresponds to the first-order polynomial filter $\hat{g}(\lambda) = 1 - \lambda$ — a low-pass filter that smooths node features across the graph.

In [ ]:
# ── Node classification with a 2-layer GCN on a toy graph ───────────────────
rng3 = np.random.default_rng(42)

# Stochastic block model: 2 communities, 10 nodes each
n_per_class = 10
n_nodes = 2 * n_per_class
A = np.zeros((n_nodes, n_nodes))

for i in range(n_per_class):
    for j in range(i+1, n_per_class):
        if rng3.random() < 0.6:  # within-community edges
            A[i, j] = A[j, i] = 1.0
        if rng3.random() < 0.6:
            A[i+n_per_class, j+n_per_class] = A[j+n_per_class, i+n_per_class] = 1.0
for i in range(n_per_class):     # cross-community edges (sparse)
    for j in range(n_per_class):
        if rng3.random() < 0.05:
            A[i, j+n_per_class] = A[j+n_per_class, i] = 1.0

# Node features: noisy class indicator
H0 = np.zeros((n_nodes, 2))
H0[:n_per_class, 0]  = 1.0 + rng3.standard_normal(n_per_class) * 0.4
H0[n_per_class:, 1]  = 1.0 + rng3.standard_normal(n_per_class) * 0.4
labels = np.array([0]*n_per_class + [1]*n_per_class)

def gcn_layer(H, A, W, activation=True):
    A_tilde = A + np.eye(len(A))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(A_tilde.sum(axis=1)))
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
    Z = A_hat @ H @ W
    return np.maximum(0, Z) if activation else Z  # ReLU

# Random weights (untrained — for illustration of message passing)
W1 = rng3.standard_normal((2, 8)) * 0.5
W2 = rng3.standard_normal((8, 2)) * 0.5

H1    = gcn_layer(H0, A, W1, activation=True)
H_out = gcn_layer(H1, A, W2, activation=False)
preds = np.argmax(H_out, axis=1)

G_sbm = nx.from_numpy_array(A)
pos_sbm = nx.spring_layout(G_sbm, seed=7)
node_colors = ['#3498db' if l == 0 else '#e74c3c' for l in labels]
pred_colors = ['#3498db' if p == 0 else '#e74c3c' for p in preds]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, cols, title in zip(axes,
        [node_colors, pred_colors],
        ['True community labels', 'GCN predictions (random weights)']):
    nx.draw_networkx_nodes(G_sbm, pos_sbm, ax=ax, node_color=cols,
                           node_size=200, alpha=0.9)
    nx.draw_networkx_edges(G_sbm, pos_sbm, ax=ax, edge_color='#aaaaaa',
                           width=0.8, alpha=0.5)
    ax.set_title(title, fontsize=11); ax.axis('off')

patches = [mpatches.Patch(color='#3498db', label='Community 0'),
           mpatches.Patch(color='#e74c3c', label='Community 1')]
axes[1].legend(handles=patches, fontsize=10)
fig.suptitle('Two-layer GCN on a stochastic block model graph', fontsize=12)
plt.tight_layout(); plt.show()
acc = (preds == labels).mean()
print(f"Accuracy with random weights: {acc:.0%}  (random chance: 50%)")
print("(Train the weights via gradient descent on cross-entropy to improve)")

<a id="part5"></a>
# Part 5: Expressivity — the Weisfeiler–Lehman Test

## 5.1 When can two graphs be distinguished?

A fundamental question: can a GNN distinguish non-isomorphic graphs? Two graphs are **isomorphic** if one can be obtained from the other by relabelling nodes. An ideal GNN would assign different representations to non-isomorphic graphs and the same representation to isomorphic ones.

**Key theorem** (Xu et al., 2019): *Any GNN using sum aggregation is at most as powerful as the Weisfeiler–Lehman (WL) graph isomorphism test.*

## 5.2 The WL test

The **1-WL test** assigns a colour (label) to each node and iteratively refines it:
1. **Initialise:** $c^{(0)}_v = h_v$ (initial node feature).
2. **Refine:** $c^{(k+1)}_v = \mathrm{HASH}\!\left(c^{(k)}_v,\; \{\!\{c^{(k)}_u : u \in \mathcal{N}(v)\}\!\}\right)$ (hash of own colour and *multiset* of neighbour colours).
3. **Compare:** at convergence, if the multisets of colours of two graphs differ, they are **not isomorphic**. If they agree, they *might* be isomorphic (WL cannot always decide).

The WL test fails on **regular graphs**: if all nodes have the same degree and the same multiset of neighbour degrees, WL assigns every node the same colour, regardless of global structure.

In [ ]:
def wl_step(colors, adj_list):
    new_colors = {}
    for v, c in colors.items():
        nbr_colors = tuple(sorted(colors[u] for u in adj_list[v]))
        new_colors[v] = hash((c, nbr_colors)) % (10**6)
    return new_colors

def run_wl(G_nx, init_color=None, n_steps=4):
    adj_list = {v: list(G_nx.neighbors(v)) for v in G_nx.nodes()}
    if init_color is None:
        colors = {v: 0 for v in G_nx.nodes()}
    else:
        colors = dict(init_color)
    history = [dict(colors)]
    for _ in range(n_steps):
        colors = wl_step(colors, adj_list)
        history.append(dict(colors))
    return history

# Two non-isomorphic graphs that WL cannot distinguish (both 3-regular, 6 nodes)
G1 = nx.Graph([(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),(0,3),(1,4),(2,5)])  # K_{3,3}
G2 = nx.Graph([(0,1),(1,2),(2,0),(3,4),(4,5),(5,3),(0,3),(1,4),(2,5)])  # prism graph

wl1 = run_wl(G1)
wl2 = run_wl(G2)

def _plot_wl(step=0):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    cmap = plt.cm.Set1
    pos1 = nx.circular_layout(G1)
    pos2 = nx.spring_layout(G2, seed=2)

    for ax, Gx, wl, pos, title in [
            (axes[0], G1, wl1, pos1, 'Graph 1  (K₃,₃)'),
            (axes[1], G2, wl2, pos2, 'Graph 2  (Prism)'),
    ]:
        cols = wl[min(step, len(wl)-1)]
        unique = sorted(set(cols.values()))
        color_map = {c: cmap(i / max(1, len(unique)-1)) for i, c in enumerate(unique)}
        node_cols  = [color_map[cols[v]] for v in Gx.nodes()]

        nx.draw_networkx_nodes(Gx, pos, ax=ax, node_color=node_cols,
                               node_size=500, alpha=0.9)
        nx.draw_networkx_edges(Gx, pos, ax=ax, edge_color='#555555', width=1.5)
        nx.draw_networkx_labels(Gx, pos, ax=ax,
                                labels={v: str(cols[v])[-3:] for v in Gx.nodes()},
                                font_size=7, font_color='white')
        multiset = sorted(cols.values())
        ax.set_title(f'{title}\nColour multiset: {multiset}', fontsize=10)
        ax.axis('off')

    fig.suptitle(f'Weisfeiler–Lehman test — step {step}  '
                 '(same colour multisets → WL cannot distinguish)', fontsize=11)
    plt.tight_layout(); plt.show()

interact(_plot_wl,
         step=IntSlider(min=0, max=4, step=1, value=0, description='Step:'));

> **Exercise 5.1.** *(WL and GNN expressivity)*
>
> **(a)** Verify by running `run_wl` that the colour multisets of $G_1$ and $G_2$ remain identical at every WL step. Confirm that the graphs are *not* isomorphic by checking their degree sequences and cycle structure.
>
> **(b)** What aggregation function $\bigoplus$ gives GNNs strictly greater expressivity than WL? (Hint: see Xu et al. 2019, Theorem 3.) Why does *mean* aggregation fail where *sum* succeeds?
>
> **(c)** Higher-order WL tests ($k$-WL for $k \ge 2$) are strictly more powerful. Describe informally what $2$-WL considers that $1$-WL does not. What is the computational cost?

<a id="part6"></a>
# Part 6: Beyond 2D — Equivariance in 3D Science

## 6.1 Molecules and the Euclidean group

A molecule is a set of atoms, each with a position $\mathbf{r}_i \in \mathbb{R}^3$ and an atom type $z_i$. A prediction of a physical property (energy, reactivity, binding affinity) must be:
- **Invariant to rotation and reflection:** $E(R\{\mathbf{r}_i\}) = E(\{\mathbf{r}_i\})$ for $R \in O(3)$.
- **Invariant to translation:** $E(\{\mathbf{r}_i + \mathbf{t}\}) = E(\{\mathbf{r}_i\})$.
- **Invariant to permutation of atoms** with the same type.

The relevant symmetry group is $E(3) = \mathbb{R}^3 \rtimes O(3)$ (rotations, reflections, translations). Architectures that respect this are called **E(3)-equivariant GNNs**.

**Examples:**
- **SchNet** (Schütt et al., 2017): uses pairwise distances $\|\mathbf{r}_i - \mathbf{r}_j\|$ as edge features — these are $E(3)$-invariant.
- **DimeNet** (Gasteiger et al., 2020): additionally uses bond angles $\theta_{ijk}$ — also $E(3)$-invariant.
- **SE(3)-Transformers** / **EGNN** (Satorras et al., 2021): pass *equivariant* vector features alongside invariant scalar features.
- **AlphaFold 2** (Jumper et al., 2021): uses frames (local coordinate systems per residue) that transform equivariantly — a key ingredient in predicting protein structure to near-experimental accuracy.

## 6.2 The blueprint of geometric deep learning

Bronstein et al. (2021) show that the following five architecture families are all instances of the same symmetry-based blueprint:

| Domain | Symmetry group $G$ | Architecture |
|--------|--------------------|-----------|
| Grids / images | $(\mathbb{Z}^2, +)$ translation | **CNN** |
| Sets | $S_n$ permutation | **Deep Sets** |
| Graphs | $S_n$ permutation | **GNN** |
| Groups | $G$ (any) | **Group CNN** |
| Manifolds / meshes | Diffeomorphism group | **Intrinsic mesh CNNs** |
| 3-D point clouds | $E(3)$ / $SE(3)$ | **E(n)-equivariant GNNs** |

The same mathematical question — *"what linear maps commute with the group action?"* — determines the layer structure in every case.

In [ ]:
# ── Visualise: distance-based features are E(3)-invariant ───────────────────
rng4 = np.random.default_rng(5)

# Small toy molecule: 4 atoms in 3D
positions = np.array([[0,0,0],[1.5,0,0],[0.75,1.3,0],[0.75,0.43,1.22]])
atom_types = ['C','C','C','H']

# Apply a random rotation R ∈ SO(3)
from scipy.spatial.transform import Rotation
R_rand = Rotation.random(random_state=3).as_matrix()
pos_rot = positions @ R_rand.T

# Pairwise distances before and after rotation
def pairwise_dists(pos):
    n = len(pos)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = np.linalg.norm(pos[i] - pos[j])
    return D

D_orig = pairwise_dists(positions)
D_rot  = pairwise_dists(pos_rot)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, D, title in [(axes[0], D_orig, 'Distance matrix\n(original positions)'),
                      (axes[1], D_rot,  'Distance matrix\n(after random rotation)'),
                      (axes[2], np.abs(D_orig - D_rot), r'$|D_\mathrm{orig} - D_\mathrm{rot}|$'+'\n(should be ≈ 0)')]:
    im = ax.imshow(D, cmap='Blues', vmin=0)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(atom_types); ax.set_yticklabels(atom_types)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{D[i,j]:.2f}', ha='center', va='center', fontsize=8)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle(r'Pairwise distances are $E(3)$-invariant: identical before and after rotation',
             fontsize=12)
plt.tight_layout(); plt.show()
print(f"Max change in distance matrix after rotation: {np.abs(D_orig - D_rot).max():.2e}")

---
# Summary

| Concept | Key formula / takeaway |
|---------|------------------------|
| **Equivariance** | $f(g\cdot\mathbf{x}) = g\cdot f(\mathbf{x})$ for all $g\in G$ |
| **CNN** | Equivariant to $(\mathbb{Z}^2,+)$ by construction (via convolution) |
| **Message passing** | $\mathbf{h}_v^{(k+1)} = \phi(\mathbf{h}_v^{(k)}, \bigoplus_{u\in\mathcal{N}(v)}\psi(\mathbf{h}_v^{(k)},\mathbf{h}_u^{(k)}))$ |
| **GCN layer** | $H^{(k+1)} = \sigma(\hat{A}H^{(k)}W^{(k)})$,  $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ |
| **Permutation equivariance** | Sum/mean aggregation is permutation-invariant; node updates are equivariant |
| **WL expressivity limit** | Any sum-GNN $\le$ WL test; WL fails on regular graphs |
| **3-D equivariance** | $E(3)$-equivariant GNNs use distances/angles as invariant features |